In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

In [2]:
sentences = [

    ("hello", "namaste"),
    ("good morning", "suprabhat"),
    ("how are you", "aap kaise ho"),
    ("i am fine", "mai theek hu"),
    ("what is your name", "tumhara naam kya hai"),
    ("my name is abhay", "mera naam abhay hai"),
    ("i love programming", "mujhe programming pasand hai"),
    ("transformers are powerful", "transformer bahut powerful hai"),
    ("deep learning is fun", "deep learning mazedar hai"),
    ("machine learning uses data", "machine learning data use karta hai"),
    ("the sun is bright", "suraj chamak raha hai"),
    ("birds can fly", "pakshi ud sakte hai"),
    ("fish live in water", "machli paani me rehti hai"),
    ("students study books", "students kitaabe padhte hai"),
    ("dogs bark loudly", "kute zor se bhokte hai"),
]

In [22]:
vocab = []
input_data = []
target = []

for src, targ in sentences:
    vocab.extend(src.split())
    vocab.extend(targ.split())

stoi = {ch:(i+1) for i,ch in enumerate(vocab)}
itos = {i+1:ch for i,ch in enumerate(vocab)}
stoi["<PAD>"] = 0
itos["<PAD>"] = 0

encode = lambda s: [stoi[i] for i in s.split()]
decode = lambda d: " ".join([itos[i] for i in d])


In [23]:
encode("hello")

[1]

In [28]:
input_data = []
target = []
for src, targ in sentences:
    # print(src,targ)
    input_data.append(torch.tensor(encode(src),dtype = torch.long))
    target.append(torch.tensor(encode(targ),dtype = torch.long))


In [30]:
input_data

[tensor([1]),
 tensor([3, 4]),
 tensor([ 6, 42,  8]),
 tensor([34, 13, 14]),
 tensor([18, 68, 20, 27]),
 tensor([26, 27, 68, 32]),
 tensor([34, 35, 38]),
 tensor([41, 42, 46]),
 tensor([52, 61, 68, 51]),
 tensor([60, 61, 58, 62]),
 tensor([66, 67, 68, 69]),
 tensor([74, 75, 76]),
 tensor([81, 82, 83, 84]),
 tensor([93, 91, 92]),
 tensor([97, 98, 99])]

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self,embed_dim,heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.heads = heads
        self.head_dim = embed_dim // heads

        self.W_q = nn.Linear(embed_dim,embed_dim)
        self.W_k = nn.Linear(embed_dim,embed_dim)
        self.W_v = nn.Linear(embed_dim,embed_dim)
        self.fc_out = nn.Linear(embed_dim,embed_dim)

    def forward(self,query,key,value,mask):

        N = query.shape[0]
        query_len, value_len , key_len = query.shape[1],value.shape[1], key.shape[1]

        query = self.W_q(query)
        key = self.W_k(key)
        value = self.W_v(value)

        # shape -> (Batch, Sequence_leng, heads, head_dim) 
        query = query.reshape(N,query_len,self.heads,self.head_dim)
        key = key.reshape(N,key_len,self.heads,self.head_dim)
        value = value.reshape(N,value_len,self.heads,self.head_dim)

        # shape -> (Batch, heads,Sequence_leng, head_dim)  (0,1,2,3)        
        query = query.transpose(1,2)
        key = key.transpose(1,2)
        value = value.transpose(1,2)

        # shape -> (Batch, heads, (T)query_len, (T)key_len)
        energy = torch.matmul(query, key.transpose(-2,-1))

        if mask is not None:
            energy = energy.masked_fill(mask == 0,-1e20)

        # shape -> (Batch, heads, (T)query_len, (T)key_len) 
        # shape -> (0,1,2,3) now dim = 2 denotes - query row wise and dim = 3 denotes columns wise key.
        weight_probs = torch.softmax(energy/(self.head_dim**0.5),dim = 3)

        # (Batch, heads, Query_len, key_len)
        attention = torch.matmul(weight_probs,value).reshape(N,query_len,heads*head_dim)
        
        #  (Batch, heads, sequence_len * head_dim)
        out = self.fc_out(out)
        return out        

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self,embed_dim,heads,dropout,forward_dim):
        super().__init__()
        self.attention = SelfAttention()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim,forward_dim*embed_dim),
            nn.ReLU(),
            nn.Linear(forward_dim*embed_dim,embed_dim)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self,query,key,value,mask):
        N = query.shape[0]
        attention = self.attention(query)
        normalized_out = self.norm1(attention+query)

        ff_out = self.feed_forward(normalized_out)
        out = self.norm2(ff_out + normalized_out)

        return out